In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
!pip install -q gdown ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.7 MB/s eta 0:00:00


In [4]:
import torch
print(torch.cuda.is_available())

True


In [5]:
!gdown 1xsx4JIkyj-Akhs5JtXaWkQLjETJQeSaF -O custom_syringe_yolo_500.zip
!unzip -o -q custom_syringe_yolo_500.zip -d /kaggle/working/

Downloading...
From (original): https://drive.google.com/uc?id=1xsx4JIkyj-Akhs5JtXaWkQLjETJQeSaF
From (redirected): https://drive.google.com/uc?id=1xsx4JIkyj-Akhs5JtXaWkQLjETJQeSaF&confirm=t&uuid=499a28c8-bb88-4088-a1bb-b4069ead44cf
To: /kaggle/working/custom_syringe_yolo_500.zip
100%|███████████████████████████████████████| 56.1M/56.1M [00:00<00:00, 152MB/s]


In [6]:
import yaml
data_yaml_path = "/kaggle/working/custom_syringe_yolo/data.yaml"
with open(data_yaml_path, "r") as f:
    cfg = yaml.safe_load(f)
cfg["path"] = "/kaggle/working/custom_syringe_yolo"
with open(data_yaml_path, "w") as f:
    yaml.dump(cfg, f)

In [7]:
from ultralytics import YOLO
model = YOLO("yolo26n.pt")
results = model.train(
    data="/kaggle/working/custom_syringe_yolo/data.yaml",
    epochs=500,
    imgsz=640,
    batch=16,
    device=0,
    mixup=0.1,
    copy_paste=0.2,
    degrees=15,
    scale=0.7,
    hsv_v=0.3,
    project="/kaggle/working/runs_syringe_500epochs",
    name="yolo26n_syringe_500epochs",
)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.123 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/custom_syringe_yolo/data.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0

In [8]:
model.export(format="onnx", imgsz=640, end2end=False)

Ultralytics 8.4.123 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 146 layers, 2,494,694 parameters, 0 gradients, 5.3 GFLOPs

PyTorch: starting from '/kaggle/working/runs_syringe_500epochs/yolo26n_syringe_500epochs/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 348ms
Prepared 2 packages in 372ms
Installed 2 packages in 16ms
 + onnxruntime==1.29.0
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 1.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.96..

'/kaggle/working/runs_syringe_500epochs/yolo26n_syringe_500epochs/weights/best.onnx'

In [9]:
import shutil
shutil.make_archive("/kaggle/working/syringe_500_final", 'zip', "/kaggle/working/runs_syringe_500epochs/yolo26n_syringe_500epochs")
print("Archive ready: /kaggle/working/syringe_500_final.zip")

Archive ready: /kaggle/working/syringe_500_final.zip
